In [ ]:
pip install -U langchain-openrouter


In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="anthropic/claude-sonnet-4.5",
    temperature=0,
    max_tokens=1024,
    max_retries=2,
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
chain = model | parser
chain.invoke("what is captial of Telangana")

"The capital of Telangana is **Hyderabad**.\n\nHyderabad serves as the capital of Telangana state in India. It's worth noting that Hyderabad also served as the joint capital of both Telangana and Andhra Pradesh for a period of 10 years (2014-2024) following the bifurcation of the former unified Andhra Pradesh state. As of June 2, 2024, Hyderabad is now exclusively the capital of Telangana."

In [ ]:
pip install langchain-community

In [ ]:
pip install -U langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# The HUGGINGFACEHUB_API_TOKEN is typically used for downloading models if not cached,
# and is usually picked up from the environment variables or implicitly.
# For 'BAAI/bge-base-en', an explicit `api_key` parameter is often not needed
# unless you're specifically interacting with the Inference API, which we are moving away from.
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
emb = embeddings.embed_query("what is captial of Telangana")

In [ ]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 10.1 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import DirectoryLoader ,  PyPDFLoader

loader = DirectoryLoader("/content/", glob="*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()

In [ ]:
from langchain_community.document_loaders import  PyPDFLoader

loader = PyPDFLoader("/content/ youtubetranscript.pdf")
docs = loader.load()
docs[0].page_content

"US  critiques  say  that  they  are  sort  of  our  bosses.  Whatever  they'll  say  we  have  to  follow.  If  \nyou\n \nare\n \na\n \ngreat\n \npower\n \nfor\n \na\n \nlong\n \ntime,\n \nyou\n \ndevelop\n \nan\n \narrogance.\n \nI\n \ndon't\n \nthink\n \nthere\n \nis\n \nanybody\n \nin\n \nthe\n \nworld\n \nwho\n \nthinks\n \nthat\n \nIndia\n \nwill\n \nbe\n \nfollowing\n \nthe\n \nUS.\n \nEven\n \nthe\n \nUS\n \ndoesn't\n \nthink.\n   \nSayyad\n \nAkbarin\n \nwas\n \nthe\n \nofficial\n \nspokesperson\n \nof\n \nIndia's\n \nministry\n \nof\n \nexternal\n \naffairs.\n   \nChina\n \nis\n \na\n \nmajor\n \nchallenge\n \nfor\n \nus.\n \nThey\n \ndo\n \nnot\n \nwant\n \nanybody\n \nelse\n \nin\n \nAsia.\n \nHe\n \nhas\n \nspent\n \nmore\n \nthan\n \n30\n \nyears\n \nstrengthening\n \nIndia's\n \nglobal\n \ndiplomacy.\n  \nChina\n \nsees\n \nIndia's\n \nrise\n \nas\n \na\n \nchallenge.\n   \nHe\n \nbreaks\n \ndown\n \nwhether\n \nthe\n \nUS\n \nand\n \nChina\n \nhave\n \nbeen\n \ninfluenc

In [ ]:
#cleaning the docs

import re

def clean_pdf_text(text):

    # 1. Remove excessive newlines
    text = text.replace("\n", " ")

    # 2. Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # 3. Fix spaced words (like "a n y")
    text = re.sub(r"(\b\w\s){2,}\w\b", lambda m: m.group(0).replace(" ", ""), text)

    # 4. Fix broken sentences spacing
    text = re.sub(r"\s([?.!,](?:\s|$))", r"\1", text)

    return text.strip()

In [ ]:
#cleaned docs
cleaned_docs = [clean_pdf_text(doc.page_content) for doc in docs]
cleaned_docs[0]

"US critiques say that they are sort of our bosses. Whatever they'll say we have to follow. If you are a great power for a long time, you develop an arrogance. I don't think there is anybody in the world who thinks that India will be following the US. Even the US doesn't think. Sayyad Akbarin was the official spokesperson of India's ministry of external affairs. China is a major challenge for us. They do not want anybody else in Asia. He has spent more than 30 years strengthening India's global diplomacy. China sees India's rise as a challenge. He breaks down whether the US and China have been influencing India's decisions. No country which is on top wants others to rise. India will rise irrespective of whether US says the curve is not going to change. How India's neighbors are being used against it. China has become a major player in our neighborhood. He cannot compete today with China because they have deeper pockets and you need to pay attention to what he's saying right now. I serv

In [ ]:
pip install -U langchain-text-splitters


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 100
)

# Use create_documents for a list of strings (cleaned_docs)
chunk = text_splitter.create_documents(cleaned_docs)
chunk[1].page_content

"They do not want anybody else in Asia. He has spent more than 30 years strengthening India's global diplomacy. China sees India's rise as a challenge. He breaks down whether the US and China have been influencing India's decisions. No country which is on top wants others to rise. India will rise irrespective of whether US says the curve is not going to change. How India's neighbors are being used against it. China has become a major player in our neighborhood. He cannot compete today with China"

In [ ]:
pip install langchain  faiss-cpu


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunk, embeddings)

In [ ]:
vectorstore.save_local("vectorDB")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 10}
)

In [ ]:
docs = retriever.invoke("what happening in iran war")

for i, d in enumerate(docs):
    print(f"\n--- Chunk {i} ---\n")
    print(d.page_content[:300])


--- Chunk 0 ---

concerned will let go of doctrine. Every country there is no country in the world which when faced with this crucial choice between doctrine and interest interests generally will generally will overcome doctrine and this is what is happening for us and there is no country in the world who hasn't don

--- Chunk 1 ---

of what is going to happen. How did that war the strikes happen? Israelis themselves have confirmed now it was because there was a target of opportunity that means somebody told them just just one day before or whatever that they are meeting so they went and struck now they could have if that opport

--- Chunk 2 ---

So you can understand that there are trends growing. Mhm. Interesting. I mean, how does how did the ceasefire happen? Like what's the process of it? So is it like America pressured us and they mediated what's the problem like how did it happen? So let's be clear in India Pakistan ties there are only

--- Chunk 3 ---

attacked than when Iran w

In [ ]:
context = "\n\n".join([doc.page_content for doc in docs])

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define the query variable
query = "what this documnet tells about?"

template = PromptTemplate(input_variables=["context" , "query"], template="""
You are a helpful assistant.

Answer ONLY using the context below.

Context:
{context}

Question:
{query}

If the answer is not in context, say "I don't know".
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()


In [ ]:
response = model.invoke(template.format(context=context, query=query))
print(response.content)

This document appears to be a transcript of a conversation or interview discussing international relations and diplomacy. Specifically, it covers:

1. **Doctrine vs. Interest in foreign policy** - How countries, including Iran, prioritize national interests over doctrine when faced with crucial choices.

2. **Military strikes and intelligence** - Discussion about Israeli strikes that happened due to a "target of opportunity" based on last-minute intelligence, and defending why certain actions weren't predicted.

3. **Ceasefire processes** - Questions about how ceasefires happen and whether America pressured or mediated.

4. **India-Pakistan relations** - Mentions that there are only two parties involved with no scope for third-party mediation, described as a "doctrinal principle."

5. **India's relations with Israel** - Discussion of growing defense, agriculture, and technology engagements between India and Israel, noting the Prime Minister has visited Israel twice versus 15 times to t